In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

# from FEATURES.features import *
from BACKTEST.backtestCalculateEVS import *
from BACKTEST.backtest import backtestTrios, calculate3LegMetrics
# from MODELS.teamInfo import *

In [2]:
import joblib


# Load NGBoost mean and variance models
mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_TEST.pkl')
variance_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_TEST.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR.pkl')

models = {
    'mean': mean_model,
    'variance': variance_model,
    'calibration_factor': calibration_factor
}

features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded NGBoost models with calibration factor: {calibration_factor}")
print(f"Number of features: {len(features)}")

Loaded NGBoost models with calibration factor: 0.95
Number of features: 128


In [3]:
pd.set_option('display.max_columns', None)

s25_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s24_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')

for df in [s25_pts,s24_pts]:
    df['MIN_MAX_LAST_10'] = df.groupby('PLAYER_ID')['MIN'].shift(1).rolling(10).max().values
    df['MIN_MIN_LAST_10'] = df.groupby('PLAYER_ID')['MIN'].shift(1).rolling(10).min().values
    df['MIN_MAX_LAST_20'] = df.groupby('PLAYER_ID')['MIN'].shift(1).rolling(20).max().values
    df['MIN_MIN_LAST_20'] = df.groupby('PLAYER_ID')['MIN'].shift(1).rolling(20).min().values
    df['MIN_MAX_LAST_40'] = df.groupby('PLAYER_ID')['MIN'].shift(1).rolling(40).max().values
    df['MIN_MIN_LAST_40'] = df.groupby('PLAYER_ID')['MIN'].shift(1).rolling(40).min().values
    
df = pd.concat([s25_pts, s24_pts]).sort_values(by='GAME_DATE')
df.sample()

,Unnamed: 0,Unnamed: 0.2,PLAYER_NAME,PLAYER_ID,MATCHUP,TEAM_ABBREVIATION,TEAM_ID,OPP_ABBREVIATION,HOME_GAME,GAME_ID,GAME_DATE,WL,PTS,AST,REB,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,STL,BLK,TOV,PLUS_MINUS,FANTASY_PTS,POINT_PER_SHOT,EFG,START_POSITION,COMMENT,E_OFF_RATING,E_DEF_RATING,NET_RATING,OREB_PCT,DREB_PCT,REB_PCT,AST_PCT,EFG_PCT,AST_TOV,USG_PCT,TS_PCT,E_PACE,PACE,PIE,POSS,PACE_PER40,E_USG_PCT,MIN,SPD,DIST,ORBC,DRBC,RBC,TCHS,SAST,FTAST,PASS,CFGM,CFGA,CFG_PCT,UFGM,UFGA,UFG_PCT,DFGM,DFGA,DFG_PCT,PTS_OFF_TOV,PTS_2ND_CHANCE,PTS_FB,PTS_PAINT,OPP_PTS_OFF_TOV,OPP_PTS_2ND_CHANCE,OPP_PTS_FB,OPP_PTS_PAINT,BLKA,PF,PFD,IS_PLAYOFF,TEAM_SEASON_ID_x,whos_favored,spread,total,team_is_favored,team_spread,OPP_BLOWOUT_RISK,TEAM_NAME,TEAM_MIN,TEAM_FGM,TEAM_FGA,TEAM_FG_PCT,TEAM_FG3M,TEAM_FG3A,TEAM_FG3_PCT,TEAM_FTM,TEAM_FTA,TEAM_FT_PCT,TEAM_OREB,TEAM_DREB,TEAM_REB,TEAM_AST,TEAM_STL,TEAM_BLK,TEAM_TOV,TEAM_PF,TEAM_PTS,TEAM_PLUS_MINUS,TEAM_PACE,GAME_PACE,OPP_PACE,OPP_TEAM_ID,TEAM_OFF_RATING,TEAM_DEF_RATING,OPP_DEF_RATING,OPP_OFF_RATING,OPP_PTS,OPP_FGM,OPP_FGA,OPP_FG_PCT,OPP_REB,OPP_AST,OPP_STL,OPP_BLK,OPP_TOV,percentageFieldGoalsAttempted2pt,percentageFieldGoalsAttempted3pt,percentagePoints2pt,percentagePointsMidrange2pt,percentagePoints3pt,percentagePointsFastBreak,percentagePointsFreeThrow,percentagePointsOffTurnovers,percentagePointsPaint,percentageAssisted2pt,percentageUnassisted2pt,percentageAssisted3pt,percentageUnassisted3pt,percentageAssistedFGM,percentageUnassistedFGM,matchupFieldGoalsMade,matchupFieldGoalsAttempted,matchupThreePointersMade,matchupThreePointersAttempted,playerPoints,matchupMinutes,DEF_FG_PCT_ALLOWED,DEF_3PT_PCT_ALLOWED,PTS_ALLOWED_PER_MIN,AGE,FREQ_FG3,FG3M_main,FG3A_main,FG3_PCT_main,NS_FG3_PCT,PLUS_MINUS_FG3,FREQ_FG2,FG2M,FG2A,FG2_PCT,NS_FG2_PCT,PLUS_MINUS_FG2,FREQ_LT_06,FGM_LT_06,FGA_LT_06,LT_06_PCT,NS_LT_06_PCT,PLUS_MINUS_LT_06,FREQ_LT_10,FGM_LT_10,FGA_LT_10,LT_10_PCT,NS_LT_10_PCT,PLUS_MINUS_LT_10,FREQ_GT_15,FGM_GT_15,FGA_GT_15,GT_15_PCT,NS_GT_15_PCT,PLUS_MINUS_GT_15,DEF_TOV_FORCED_PER_MIN,DEF_BLOCKS_PER_MIN,DEF_SHOOTING_FOULS_PER_MIN,DEF_AST_ALLOWED_PER_MIN,STARTING,GAMES_THIS_SEASON,HEIGHT,WEIGHT,GUARD,FORWARD,CENTER,TEAM_DAYS_REST,PLAYER_DAYS_REST,TEAM_B2B,IS_BACK_TO_BACK,PLAYER_MISSED_LAST,PTS_ROLLING_AVG_5,AST_ROLLING_AVG_5,FGM_ROLLING_AVG_5,FGA_ROLLING_AVG_5,FG_PCT_ROLLING_AVG_5,FG3M_ROLLING_AVG_5,FG3A_ROLLING_AVG_5,FG3_PCT_ROLLING_AVG_5,FTA_ROLLING_AVG_5,FTM_ROLLING_AVG_5,FT_PCT_ROLLING_AVG_5,TOV_ROLLING_AVG_5,TS_PCT_ROLLING_AVG_5,USG_PCT_ROLLING_AVG_5,MIN_ROLLING_AVG_5,PACE_ROLLING_AVG_5,PIE_ROLLING_AVG_5,E_OFF_RATING_ROLLING_AVG_5,NET_RATING_ROLLING_AVG_5,TCHS_ROLLING_AVG_5,POSS_ROLLING_AVG_5,EFG_PCT_ROLLING_AVG_5,percentagePointsPaint_ROLLING_AVG_5,percentagePointsMidrange2pt_ROLLING_AVG_5,percentagePoints3pt_ROLLING_AVG_5,PTS_ROLLING_AVG_10,AST_ROLLING_AVG_10,FGM_ROLLING_AVG_10,FGA_ROLLING_AVG_10,FG_PCT_ROLLING_AVG_10,FG3M_ROLLING_AVG_10,FG3A_ROLLING_AVG_10,FG3_PCT_ROLLING_AVG_10,FTA_ROLLING_AVG_10,FTM_ROLLING_AVG_10,FT_PCT_ROLLING_AVG_10,TOV_ROLLING_AVG_10,TS_PCT_ROLLING_AVG_10,USG_PCT_ROLLING_AVG_10,MIN_ROLLING_AVG_10,PACE_ROLLING_AVG_10,PIE_ROLLING_AVG_10,E_OFF_RATING_ROLLING_AVG_10,NET_RATING_ROLLING_AVG_10,TCHS_ROLLING_AVG_10,POSS_ROLLING_AVG_10,EFG_PCT_ROLLING_AVG_10,percentagePointsPaint_ROLLING_AVG_10,percentagePointsMidrange2pt_ROLLING_AVG_10,percentagePoints3pt_ROLLING_AVG_10,PTS_ROLLING_AVG_40,AST_ROLLING_AVG_40,FGM_ROLLING_AVG_40,FGA_ROLLING_AVG_40,FG_PCT_ROLLING_AVG_40,FG3M_ROLLING_AVG_40,FG3A_ROLLING_AVG_40,FG3_PCT_ROLLING_AVG_40,FTA_ROLLING_AVG_40,FTM_ROLLING_AVG_40,FT_PCT_ROLLING_AVG_40,TOV_ROLLING_AVG_40,TS_PCT_ROLLING_AVG_40,USG_PCT_ROLLING_AVG_40,MIN_ROLLING_AVG_40,PACE_ROLLING_AVG_40,PIE_ROLLING_AVG_40,E_OFF_RATING_ROLLING_AVG_40,NET_RATING_ROLLING_AVG_40,TCHS_ROLLING_AVG_40,POSS_ROLLING_AVG_40,EFG_PCT_ROLLING_AVG_40,percentagePointsPaint_ROLLING_AVG_40,percentagePointsMidrange2pt_ROLLING_AVG_40,percentagePoints3pt_ROLLING_AVG_40,GAMES_VS_OPP,MATCHUP_AVG_PTS_TO_DATE,PTS

In [4]:
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')
backtestData = backtestData[(backtestData['BOOKMAKER'] == 'underdog') &  (backtestData['CATEGORY'] == 'player_points')]
backtestData.sample(5)

/var/folders/nw/9w4r5hrd05s122kt5hg_t8bw0000gn/T/ipykernel_54856/3488731641.py:1: DtypeWarning: Columns (11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')


,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,NAME,CATEGORY,BOOKMAKER,SIDE,LINE,ODDS,HOME_TEAM,AWAY_TEAM,game_id,commence_time,GAME_DATE,period_id,fair_line,fair_odds,OVER/ODDS
14808,14808,14808,14808,Jrue Holiday,player_points,underdog,over,11.5,-137,Boston Celtics,Toronto Raptors,c5d4bea033d5df350953050bdcb5dbf6,2024-11-17T01:10:00Z,2024-11-16,NaN,NaN,NaN,over
34529,34529,34529,34529,Jaden Ivey,player_points,underdog,under,16.5,-137,Denver Nuggets,Detroit Pistons,efb1af33c4ee8c1152e9f4c8fa696aed,2024-12-29T02:10:00Z,2024-12-28,NaN,NaN,NaN,under
161954,161954,161954,73893,Jaren Jackson Jr,player_points,underdog,under,20.5,-137,Oklahoma City Thunder,Memphis Grizzlies,NaN,NaN,2025-03-28,game,26.5,108.0,under
38394,38394,38394,38394,Obi Toppin,player_points,underdog,over,8.5,-137,Indiana Pacers,Phoenix Suns,7c138909a671e2895ea58542823dde19,2025-01-05T00:10:00Z,2025-01-04,NaN,NaN,NaN,over
129225,129225,129225,31746,Jabari Smith,player_points,underdog,under,12.5,-137,Houston Rockets,Phoenix Suns,NaN,NaN,2025-03-13,game,10.5,-110.0,under


In [5]:
date = '2025-03-22'

results = backtestTrios(
    data=df,
    backtestData=backtestData,
    gameDate=date,
    models=models,  
    features=features,
    edge_threshold=0.20, 
    top_n=10, 
    variance_inflation=1.1, 
    distribution_type='t', stat_col='PTS', 
    use_monte_carlo=True, n_simulations=10000, max_kelly=0.25, stake=10)

# Comprehensive results analysis
if not results.empty:
    print("=" * 60)
    print("TRIO BETTING RESULTS ANALYSIS")
    print("=" * 60)
    
    # Overall metrics
    total_bets = len(results)
    total_wins = results['parlay_won'].sum()
    win_rate = results['parlay_won'].mean()
    
    # Calculate profit/loss
    total_profit = results['parlay_profit'].sum()
    avg_profit = results['parlay_profit'].mean()
    
    print(f"Total Bets: {total_bets}")
    print(f"Wins: {total_wins}")
    print(f"Win Rate: {win_rate:.2%}")
    print(f"Total Profit: ${total_profit:.2f}")
    print(f"Average Profit per Bet: ${avg_profit:.2f}")

results

TRIO BETTING RESULTS ANALYSIS
Total Bets: 10
Wins: 0
Win Rate: 0.00%
Total Profit: $-100.00
Average Profit per Bet: $-10.00


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,PROB 1,PROB 2,PROB 3,PROB ALL THREE,EDGE 1,EDGE 2,EDGE 3,COMBINED EDGE,EV$,KELLY FULL,RECOMMENDATION,CONFIDENCE INTERVAL 1,CONFIDENCE INTERVAL 2,CONFIDENCE INTERVAL 3,INTERVAL WIDTH 1,INTERVAL WIDTH 2,INTERVAL WIDTH 3,SIGMA 1,SIGMA 2,SIGMA 3,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3,EXPECTED ROI,SIMULATION METHOD,actual1,actual2,actual3,leg1_won,leg2_won,leg3_won,parlay_won,parlay_recommendation,parlay_profit,date
0,Trendon Watford,Trae Young,Mouhamed Gueye,11.5,24.5,4.5,8.92,28.85,8.44,under,over,over,0.710,0.716,0.788,0.4005,0.132,0.138,0.21,0.207,14.03,0.281,1,"(0.0, 20.1)","(11.2, 46.5)","(0.0, 19.8)",20.11,35.26,19.77,5.71,9.00,5.78,Med,High,Med,140.3,Monte Carlo,26,25,2,0,1,0,0,1,-10.0,2025-03-22
1,Trae Young,Moses Moody,Mouhamed Gueye,24.5,12.5,4.5,28.85,9.91,8.44,over,under,over,0.716,0.702,0.788,0.3960,0.138,0.124,0.21,0.203,13.76,0.275,1,"(11.2, 46.5)","(0.0, 21.7)","(0.0, 19.8)",35.26,21.69,19.77,9.00,6.01,5.78,High,High,Med,137.6,Monte Carlo,25,20,2,1,0,0,0,1,-10.0,2025-03-22
2,Trendon Watford,Moses Moody,Mouhamed Gueye,11.5,12.5,4.5,8.92,9.91,8.44,under,under,over,0.710,0.702,0.788,0.3930,0.132,0.124,0.21,0.200,13.58,0.272,0,"(0.0, 20.1)","(0.0, 21.7)","(0.0, 19.8)",20.11,21.69,19.77,5.71,6.01,5.78,Med,High,Med,135.8,Monte Carlo,26,20,2,0,0,0,0,0,-10.0,2025-03-22
3,Trendon Watford,Zaccharie Risacher,Mouhamed Gueye,11.5,11.5,4.5,8.92,14.64,8.44,under,over,over,0.710,0.701,0.788,0.3924,0.132,0.123,0.21,0.199,13.54,0.271,0,"(0.0, 20.1)","(1.0, 28.3)","(0.0, 19.8)",20.11,27.28,19.77,5.71,6.96,5.78,Med,High,Med,135.4,Monte Carlo,26,14,2,0,1,0,0,0,-10.0,2025-03-22
4,Moses Moody,Zaccharie Risacher,Mouhamed Gueye,12.5,11.5,4.5,9.91,14.64,8.44,under,over,over,0.702,0.701,0.788,0.3879,0.124,0.123,0.21,0.195,13.28,0.266,0,"(0.0, 21.7)","(1.0, 28.3)","(0.0, 19.8)",21.69,27.28,19.77,6.01,6.96,5.78,High,High,Med,132.8,Monte Carlo,20,14,2,0,1,0,0,0,-10.0,2025-03-22
5,Tyrese Martin,Trae Young,Mouhamed Gueye,11.5,24.5,4.5,9.12,28.85,8.44,under,over,over,0.683,0.716,0.788,0.3854,0.105,0.138,0.21,0.192,13.12,0.262,0,"(0.0, 21.3)","(11.2, 46.5)","(0.0, 19.8)",21.34,35.26,19.77,6.24,9.00,5.78,High,High,Med,131.2,Monte Carlo,9,25,2,1,1,0,0,0,-10.0,2025-03-22
6,Tyrese Martin,Trendon Watford,Mouhamed Gueye,11.5,11.5,4.5,9.12,8.92,8.44,under,under,over,0.683,0.710,0.788,0.3825,0.105,0.132,0.21,0.189,12.95,0.259,0,"(0.0, 21.3)","(0.0, 20.1)","(0.0, 19.8)",21.34,20.11,19.77,6.24,5.71,5.78,High,Med,Med,129.5,Monte Carlo,9,26,2,1,0,0,0,0,-10.0,2025-03-22
7,Myles Turner,Trae Young,Mouhamed Gueye,14.5,24.5,4.5,12.24,28.85,8.44,under,over,over,0.678,0.716,0.788,0.3825,0.100,0.138,0.21,0.189,12.95,0.259,0,"(0.3, 24.2)","(11.2, 46.5)","(0.0, 19.8)",23.82,35.26,19.77,6.08,9.00,5.78,High,High,Med,129.5,Monte Carlo,22,25,2,0,1,0,0,0,-10.0,2025-03-22
8,Tyrese Haliburton,Trae Young,Mouhamed Gueye,16.5,24.5,4.5,14.06,28.85,8.44,under,over,over,0.677,0.716,0.788,0.3817,0.099,0.138,0.21,0.189,12.90,0.258,0,"(1.1, 27.0)","(11.2, 46.5)","(0.0, 19.8)",25.93,35.26,19.77,6.62,9.00,5.78,High,High,Med,129.0,Monte Carlo,16,25,2,1,1,0,0,0,-10.0,2025-03-22
9,Noah Clowney,Trae Young,Mouhamed Gueye,9.5,24.5,4.5,7.34,28.85,8.44,under,over,over,0.677,0.716,0.788,0.3815,0.099,0.138,0.21,0.188,12.89,0.258,0,"(0.0, 18.9)","(11.2, 46.5)","(0.0, 19.8)",18.87,35.26,19.77,5.88,9.00,5.78,Med,High,Med,128.9,Monte Carlo,5,25,2,1,1,0,0,0,-10.0,2025-03-22


In [6]:
uniqueDates = sorted(s25_pts['GAME_DATE'].unique())
all_results = []

for i, date in enumerate(uniqueDates, 1):
    print(f"Processing {date}... {i}/{len(uniqueDates)}")
    
    try:
        date_results = backtestTrios(
        data=df,
        backtestData=backtestData,
        gameDate=date,
        models=models,  
        features=features,
        edge_threshold=0.12, 
        top_n=10, 
        variance_inflation=1.1, 
        distribution_type='t', stat_col='PTS', 
        use_monte_carlo=False, n_simulations=10000, max_kelly=0.25, stake=100)
        
        if not date_results.empty:
            all_results.append(date_results)
            print(f"  ✓ Processed {len(date_results)} bets")
        else:
            print(f"  No results for {date}")
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        continue

# Combine all results
if all_results:
    final_results = pd.concat(all_results, ignore_index=True)
    print(f"\n✓ Complete! Total bets analyzed: {len(final_results)}")

Processing 2024-10-22... 1/163
  ✓ Processed 10 bets
Processing 2024-10-23... 2/163
  ✓ Processed 10 bets
Processing 2024-10-24... 3/163
  ✓ Processed 10 bets
Processing 2024-10-25... 4/163


: 

: 

In [ ]:
all_metrics = calculate3LegMetrics(final_results, stake=5)

# Calculate metrics for recommended bets only
recommended = final_results[final_results['parlay_recommendation'] == 1]
rec_metrics = calculate3LegMetrics(recommended, stake=5)

# Display comprehensive results
print("=" * 60)
print("           COMPREHENSIVE BETTING ANALYSIS")
print("=" * 60)

if all_metrics:
    print(f"\n ALL BETS ANALYSIS:")
    print(f"   Total Bets: {all_metrics['total_bets']:,}")
    print(f"   Wins: {all_metrics['total_wins']:,}")
    print(f"   Hit Rate: {all_metrics['win_rate']:.2%}")
    print(f"   Total Profit: ${all_metrics['total_profit']:,.2f}")
    print(f"   Total Staked: ${all_metrics['total_staked']:,.2f}")
    print(f"   ROI: {all_metrics['roi_percent']:.2f}%")
    print(f"   Volatility: ${all_metrics['volatility']:.2f}")
    print(f"   Max Drawdown: ${all_metrics['max_drawdown']:.2f}")
    print(f"   Sharpe Ratio: {all_metrics['sharpe_ratio']:.3f}")
    
    # Display probability metrics
    if 'probability_metrics' in all_metrics and 'error' not in all_metrics['probability_metrics']:
        prob_metrics = all_metrics['probability_metrics']
        print(f"\n   PROBABILITY METRICS:")
        print(f"   Brier Score: {prob_metrics.get('brier_score', 'N/A'):.4f} (lower is better)")
        print(f"   Log Loss: {prob_metrics.get('log_loss', 'N/A'):.4f} (lower is better)")
        if not np.isnan(prob_metrics.get('auc_roc', np.nan)):
            print(f"   AUC-ROC: {prob_metrics.get('auc_roc', 'N/A'):.4f} (higher is better)")
        else:
            print(f"   AUC-ROC: NaN ({prob_metrics.get('auc_roc_note', 'Only one class present')})")
        print(f"   Samples: {prob_metrics.get('n_samples', 'N/A')}")

if rec_metrics:
    print("\n RECOMMENDED BETS ANALYSIS (Edge > 0.20):")
    print(f"   Total Bets: {rec_metrics['total_bets']:,}")
    print(f"   Wins: {rec_metrics['total_wins']:,}")
    print(f"   Hit Rate: {rec_metrics['win_rate']:.2%}")
    print(f"   Total Profit: ${rec_metrics['total_profit']:,.2f}")
    print(f"   Total Staked: ${rec_metrics['total_staked']:,.2f}")
    print(f"   ROI: {rec_metrics['roi_percent']:.2f}%")
    print(f"   Volatility: ${rec_metrics['volatility']:.2f}")
    print(f"   Max Drawdown: ${rec_metrics['max_drawdown']:.2f}")
    print(f"   Sharpe Ratio: {rec_metrics['sharpe_ratio']:.3f}")
    
    # Display probability metrics for recommended bets
    if 'probability_metrics' in rec_metrics and 'error' not in rec_metrics['probability_metrics']:
        prob_metrics = rec_metrics['probability_metrics']
        print(f"\n   PROBABILITY METRICS:")
        print(f"   Brier Score: {prob_metrics.get('brier_score', 'N/A'):.4f} (lower is better)")
        print(f"   Log Loss: {prob_metrics.get('log_loss', 'N/A'):.4f} (lower is better)")
        if not np.isnan(prob_metrics.get('auc_roc', np.nan)):
            print(f"   AUC-ROC: {prob_metrics.get('auc_roc', 'N/A'):.4f} (higher is better)")
        else:
            print(f"   AUC-ROC: NaN ({prob_metrics.get('auc_roc_note', 'Only one class present')})")
        print(f"   Samples: {prob_metrics.get('n_samples', 'N/A')}")

# Plots cumulative profit over time
if all_metrics and not all_metrics['daily_pnl'].empty:
    plt.figure(figsize=(14, 8))
    
    plt.subplot(2, 1, 1)
    plt.plot(all_metrics['daily_pnl']['date'], all_metrics['daily_pnl']['cumulative_profit'], 
             linewidth=2, label='All Bets', color='orange')
    if rec_metrics and not rec_metrics['daily_pnl'].empty:
        plt.plot(rec_metrics['daily_pnl']['date'], rec_metrics['daily_pnl']['cumulative_profit'], 
                 linewidth=2, label='Recommended Bets', color='purple')
    plt.title('Cumulative Profit Over Time', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Cumulative Profit ($)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 1, 2)
    plt.bar(all_metrics['daily_pnl']['date'], all_metrics['daily_pnl']['profit'], 
            alpha=0.7, label='Daily P&L', color='green')
    plt.title('Daily Profit/Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Daily P&L ($)')
    plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
